# Task 2: Create a SHAP Explainer

This notebook follows a basic approach: prepare the data, train a Random Forest model, and create a SHAP explainer.

## 1. Install and import libraries

In [1]:
!pip install -q shap imbalanced-learn

In [2]:
import pandas as pd
import shap
from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import RandomOverSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

## 2. Upload the dataset

In [3]:
uploaded = files.upload()
file_name = next(iter(uploaded))
data = pd.read_csv(file_name)

print(data.shape)
data.head()

Saving healthcare-dataset-stroke-data.csv to healthcare-dataset-stroke-data.csv
(5110, 12)


,id,gender,age,hypertension,heart_disease,ever_married,work_type,Residence_type,avg_glucose_level,bmi,smoking_status,stroke
0,9046,Male,67.0,0,1,Yes,Private,Urban,228.69,36.6,formerly smoked,1
1,51676,Female,61.0,0,0,Yes,Self-employed,Rural,202.21,NaN,never smoked,1
2,31112,Male,80.0,0,1,Yes,Private,Rural,105.92,32.5,never smoked,1
3,60182,Female,49.0,0,0,Yes,Private,Urban,171.23,34.4,smokes,1
4,1665,Female,79.0,1,0,Yes,Self-employed,Rural,174.12,24.0,never smoked,1


## 3. Preprocess the data

Convert categorical columns into numeric columns, fill missing BMI values with 0, and remove the ID column. These are the preprocessing steps used in the lecture's `utils.py`.

In [4]:
categorical_cols = ['gender', 'ever_married', 'work_type',
                    'Residence_type', 'smoking_status']

encoded = pd.get_dummies(data[categorical_cols], prefix=categorical_cols)
data = pd.concat([encoded, data], axis=1)
data.drop(categorical_cols, axis=1, inplace=True)
data['bmi'] = data['bmi'].fillna(0)
data.drop(['id'], axis=1, inplace=True)

data.head()

,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,work_type_Govt_job,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,...,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke
0,False,True,False,False,True,False,False,True,False,False,...,False,True,False,False,67.0,0,1,228.69,36.6,1
1,True,False,False,False,True,False,False,False,True,False,...,False,False,True,False,61.0,0,0,202.21,0.0,1
2,False,True,False,False,True,False,False,True,False,False,...,False,False,True,False,80.0,0,1,105.92,32.5,1
3,True,False,False,False,True,False,False,True,False,False,...,False,False,False,True,49.0,0,0,171.23,34.4,1
4,True,False,False,False,True,False,False,False,True,False,...,False,False,True,False,79.0,1,0,174.12,24.0,1


## 4. Separate features and target

The target column is `stroke`. The other columns are used as features.

In [5]:
X = data.drop('stroke', axis=1)
y = data['stroke']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=2021
)

print('Training data:', X_train.shape)
print('Testing data:', X_test.shape)

Training data: (4088, 21)
Testing data: (1022, 21)


## 5. Balance the training data

The dataset has fewer stroke cases than non-stroke cases. Random oversampling increases the number of minority-class examples in the training data.

In [6]:
oversampler = RandomOverSampler(sampling_strategy='minority', random_state=2021)
X_train_balanced, y_train_balanced = oversampler.fit_resample(X_train, y_train)

print(y_train_balanced.value_counts())

stroke
0    3889
1    3889
Name: count, dtype: int64


## 6. Train a Random Forest model

In [7]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train_balanced, y_train_balanced)

print('Model training completed.')

Model training completed.


## 7. Create the SHAP explainer

TreeExplainer is suitable for tree-based models such as Random Forest.

In [8]:
explainer = shap.TreeExplainer(model)

print('SHAP explainer created.')

SHAP explainer created.


## 8. Calculate SHAP values

Here, we calculate SHAP values for a small number of test examples. These values show how each feature contributes to the model's prediction.

In [9]:
sample = X_test.iloc[:10]
shap_values = explainer.shap_values(sample)

print('SHAP values calculated.')
print('Number of examples:', len(sample))

SHAP values calculated.
Number of examples: 10


## 9. Check the model on the test set

The positive class is stroke = 1. Accuracy can look high because most patients did not have a stroke, so recall, F1, and the confusion matrix are also printed.

In [10]:
y_pred = model.predict(X_test)
y_score = model.predict_proba(X_test)[:, 1]

print('Classes:', model.classes_)
print('Accuracy:', accuracy_score(y_test, y_pred))
print('Balanced accuracy:', balanced_accuracy_score(y_test, y_pred))
print('Precision for stroke = 1:', precision_score(y_test, y_pred, pos_label=1))
print('Recall for stroke = 1:', recall_score(y_test, y_pred, pos_label=1))
print('F1 for stroke = 1:', f1_score(y_test, y_pred, pos_label=1))
print('ROC AUC:', roc_auc_score(y_test, y_score))
print('Average precision:', average_precision_score(y_test, y_score))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

Classes: [0 1]
Accuracy: 0.9432485322896281
Balanced accuracy: 0.5243415637860083
Precision for stroke = 1: 0.21428571428571427
Recall for stroke = 1: 0.06
F1 for stroke = 1: 0.09375
ROC AUC: 0.7607510288065844
Average precision: 0.1377712915484563
Confusion matrix:
[[961  11]
 [ 47   3]]


## 10. Check the SHAP explainer

TreeExplainer is used because the model is a Random Forest. The values below are for the probability of stroke = 1. For one test row, the base value plus the SHAP values should be close to the model probability.

In [11]:
one_row = X_test.iloc[[0]]
one_values = explainer.shap_values(one_row)

if isinstance(one_values, list):
    stroke_values = one_values[1][0]
    base_value = explainer.expected_value[1]
elif len(one_values.shape) == 3:
    stroke_values = one_values[0, :, 1]
    base_value = explainer.expected_value[1]
else:
    stroke_values = one_values[0]
    base_value = explainer.expected_value

print('Explainer: TreeExplainer')
print('Explained output: probability of stroke = 1')
print('Base value:', base_value)
print('Base value + SHAP sum:', base_value + stroke_values.sum())
print('Model probability:', model.predict_proba(one_row)[0, 1])

Explainer: TreeExplainer
Explained output: probability of stroke = 1
Base value: 0.49949858575469264
Base value + SHAP sum: 0.07000000000000095
Model probability: 0.07


## Task 2 completed

We prepared the stroke dataset, trained a Random Forest model, and created a SHAP TreeExplainer. The SHAP values are ready to be visualised in the next tasks.